# Advanced Feature Engineering

In diesem Notebook werden zusätzliche Features erstellt, um die Vorhersage des Wettmarktes "Both Teams To Score (BTTS)" zu verbessern.

Ziel ist es, aus den historischen Spieldaten weitere aussagekräftige Merkmale abzuleiten.

Bibliotheken

In [1]:
import pandas as pd 
import numpy as np 

In [2]:
matches = pd.read_csv("../data/processed/btts_features.csv")

matches.head()

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,LBCA,HomeGoalsLast5,AwayGoalsLast5,HomeConcededLast5,AwayConcededLast5,HomePoints,AwayPoints,HomeFormLast5,AwayFormLast5,BTTS
0,E0,2021-08-28,17:30,Liverpool,Chelsea,1,1,D,1,1,...,NaN,2.0,2.0,0.0,0.0,1,1,3.0,3.0,1
1,E0,2021-08-28,12:30,Man City,Arsenal,5,0,H,3,0,...,NaN,5.0,0.0,0.0,2.0,3,0,3.0,0.0,0
2,E0,2021-08-28,15:00,Aston Villa,Brentford,1,1,D,1,1,...,NaN,2.0,0.0,0.0,0.0,1,1,3.0,1.0,1
3,E0,2021-08-28,15:00,Brighton,Everton,0,2,A,0,1,...,NaN,2.0,2.0,0.0,2.0,0,3,3.0,1.0,0
4,E0,2021-08-28,15:00,Newcastle,Southampton,2,2,D,0,0,...,NaN,2.0,1.0,4.0,3.0,1,1,0.0,0.0,1


In [ ]:
# Feature Offensivestärke

matches["HomeShotsOnTargetLast5"] = (
    matches.groupby("HomeTeam")["HST"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

matches["AwayShotsOnTargetLast5"] = (
    matches.groupby("AwayTeam")["AST"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

matches[
    [
        "HomeTeam",
        "AwayTeam",
        "HST",
        "AST",
        "HomeShotsOnTargetLast5",
        "AwayShotsOnTargetLast5"
    ]
].tail(10)

C:\Users\HP\AppData\Local\Temp\ipykernel_32760\3140415477.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches["HomeShotsOnTargetLast5"] = (
C:\Users\HP\AppData\Local\Temp\ipykernel_32760\3140415477.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches["AwayShotsOnTargetLast5"] = (


,HomeTeam,AwayTeam,HST,AST,HomeShotsOnTargetLast5,AwayShotsOnTargetLast5
1856,Nott'm Forest,Bournemouth,5,4,3.6,3.8
1857,Man City,Aston Villa,3,5,6.6,4.0
1858,Sunderland,Chelsea,2,1,3.2,3.6
1859,Liverpool,Brentford,8,2,4.4,3.4
1860,Tottenham,Everton,6,3,4.2,4.8
1861,Crystal Palace,Arsenal,3,7,3.4,4.0
1862,Burnley,Wolves,8,4,3.4,3.4
1863,Brighton,Man United,2,7,6.4,3.0
1864,Fulham,Newcastle,6,2,4.6,4.8
1865,West Ham,Leeds,9,3,3.4,4.0


Feature BTTS-Historie der Teams

In [5]:
matches["BTTS"] = (
    (matches["FTHG"] > 0) &
    (matches["FTAG"] > 0)
).astype(int)

In [6]:
#wie oft hatte die Heimmannschaft in den letzen 5 Heimspielen BTTS

matches["HomeBTTSLast5"] = (
    matches.groupby("HomeTeam")["BTTS"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

C:\Users\HP\AppData\Local\Temp\ipykernel_32760\470639832.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches["HomeBTTSLast5"] = (


In [8]:
# Dasselbe für Auswärtsteams

matches["AwayBTTSLast5"] = (
    matches.groupby("AwayTeam")["BTTS"]
    .transform(lambda x: x. shift(1).rolling(5, min_periods=1).mean())
)

C:\Users\HP\AppData\Local\Temp\ipykernel_32760\494716554.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches["AwayBTTSLast5"] = (


In [9]:
matches[
    [
        "HomeTeam",
        "AwayTeam",
        "HomeBTTSLast5",
        "AwayBTTSLast5",
        "BTTS"
    ]
].tail(10)

,HomeTeam,AwayTeam,HomeBTTSLast5,AwayBTTSLast5,BTTS
1856,Nott'm Forest,Bournemouth,0.6,0.4,1
1857,Man City,Aston Villa,0.6,0.6,1
1858,Sunderland,Chelsea,0.2,0.6,1
1859,Liverpool,Brentford,0.8,0.4,1
1860,Tottenham,Everton,0.8,0.8,0
1861,Crystal Palace,Arsenal,0.4,0.6,1
1862,Burnley,Wolves,0.4,0.2,1
1863,Brighton,Man United,0.4,0.4,0
1864,Fulham,Newcastle,0.4,0.6,0
1865,West Ham,Leeds,0.4,0.8,0


In [10]:
matches.to_csv("../data/processed/btts_features_v2.csv", index=False)

print("Datensatz gespeichert.")

Datensatz gespeichert.
